# **Final Project - Development and Evaluation of Baseline, Enhanced and Hierarchical Agentic Retrieval Augmented Generation (RAG) Architectures for Improving Large Language Models**


##
###**Rice University**





#Major contributions:
###**Author: Gomathi Ramesh**

- **Pretrained Transformer based encoder and decoder models setup for project:**
  - pretrained encoder BGE-M3
  - indexing and storing them in FAISS
  - using a pretrained decoder model Qwen3-4B-Instruct to generate the final answer

- **Model Optimization techniques & Training Strategies**

   - The objective of **Fine_Tuning** the pretrained model is to make Qwen return short, correct, context-grounded answers for the HotpotQA dataset, ​improving its performance in a RAG setting.

   - In causal language modeling, the prompt and ground-truth answer are concatenated, enabling implicit application of **teacher forcing** while optimizing the model using cross-entropy loss.


- **The Fine Tuning methodology full pipeline**

   - applied Supervised Fine-Tuning (SFT) to the pretrained Qwen3-4B-Instruct model using Low-Rank Adaptation (LoRA) for parameter-efficient training.
   - Training samples were constructed from HotpotQA supporting facts and formatted as prompt–answer pairs.
   - Low-Rank Adaptation (LoRA) adapters are trained while the base Qwen model weights remain frozen.

- **RAG - ARCHITECTURES IMPLEMENTATION  PHASE**            

    - Implementation of Baseline RAG full pipe line on pretrained models
    - Implementation of Baseline RAG full pipe line on Fine-tuned QWEN pretrained models.

- **Hyperparameter Tuning on Pretrained Baseline RAG**

   -  the optimal configuration is selected as
      - k = 3
      -  temperature = 0.3
      - max_new_tokens = 20,
      
  Those optimal values used for subsequent evaluation and comparison for all architectures.


- **Comparison:Pretrained vs Fine-Tuned Generator
Evaluation on Validation Set:**

     - The baseline RAG model was evaluated on the validation set using the optimal hyperparameters (k = 3, temperature = 0.3, max_new_tokens = 20) to compare the performance of pretrained and fine-tuned generator models and proved Fine-Tuned Model Outperformance






### Question answering using HotpotQA dataset

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install datasets

In [ ]:
# Basic libraries
import os
import re
import numpy as np
import pandas as pd

# PyTorch
import torch

# Hugging Face dataset utilities
from datasets import load_dataset, concatenate_datasets

# Transformer models and tokenizers
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# Vector database
import faiss

# progress bars
from tqdm import tqdm

###Load HotpotQA question, answer, context, and supporting-fact fields

In [ ]:
from datasets import load_dataset

dataset_HotpotQA = load_dataset("hotpot_qa", "distractor")
print(dataset_HotpotQA)

## Data Split
###Train / validation / test workflow

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Combine original train and validation
full_dataset = concatenate_datasets([dataset_HotpotQA["train"], dataset_HotpotQA["validation"]])

# First split: 80% train, 20% temp
split1 = full_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split1["train"]

# Second split: split remaining 20% into 10% val + 10% test
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

val_data = split2["train"]
test_data = split2["test"]

print(len(train_data), len(val_data), len(test_data))

In [ ]:
print(len(full_dataset))

#Baseline RAG Preprocessing

- Manual preprocessing: cleaning, flattening, chunking
- Model tokenizer preprocessing = subword tokenization automatically handled by the pretrained tokenizer

###Flatten HotpotQA context

In [ ]:
sample = train_data[0]

titles = sample["context"]["title"]
sentences = sample["context"]["sentences"]

context_text = ""

for title, sent_list in zip(titles, sentences):
    context_text += title + ": " + " ".join(sent_list) + "\n\n"

print("Question:", sample["question"])
print("\nAnswer:", sample["answer"])
print("\nContext:\n", context_text[:1000])

In [ ]:
print("Question:", sample["question"])
print("Answer:", sample["answer"])
print("Context:", context_text)

In [ ]:
sample = train_data[0]

print(type(sample))
print(sample.keys())
print(type(sample["context"]))
print(sample["context"])

###Function - converts each HotpotQA example into document chunks

###Converted each document into retrieval-ready chunks

In [ ]:
def build_document_chunks(example):
    chunks = []

    titles = example["context"]["title"]
    sentences = example["context"]["sentences"]

    for title, sent_list in zip(titles, sentences):
        chunk_text = title + ": " + " ".join(sent_list)
        chunks.append(chunk_text)

    return {
        "question": example["question"],
        "answer": example["answer"],
        "chunks": chunks,
        "supporting_facts": example["supporting_facts"]
    }

In [ ]:
processed_sample = build_document_chunks(train_data[0])

print("Question:", processed_sample["question"])
print("Answer:", processed_sample["answer"])
print("Number of chunks:", len(processed_sample["chunks"]))
print("\nFirst chunk:\n", processed_sample["chunks"][0])

In [ ]:
train_processed = train_data.map(build_document_chunks)
val_processed = val_data.map(build_document_chunks)
test_processed = test_data.map(build_document_chunks)

In [ ]:
sample_processed = train_processed[0]

print(sample_processed.keys())
print("Question:", sample_processed["question"])
print("Answer:", sample_processed["answer"])
print("Number of chunks:", len(sample_processed["chunks"]))
print("\nFirst chunk:\n", sample_processed["chunks"][0])
print("\nSupporting facts:\n", sample_processed["supporting_facts"])

In [ ]:
print(type(train_processed[0]["chunks"]))
print(len(train_processed[0]["chunks"]))
print(train_processed[0]["chunks"][0][:300])

##Load BGE-M3 and test embedding on a small sample of chunks
Its tokenizer handles the tokenization needed before embedding

In [ ]:


model_name = "BAAI/bge-m3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

In [ ]:
sample_chunks = train_processed[0]["chunks"][:3]

inputs = tokenizer(
    sample_chunks,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

#set up the Baseline RAG pipeline

- embed chunks with BGE-M3
- store embeddings in FAISS
- embed question
- retrieve top-k chunks
- send retrieved context and question to Qwen3
- generate answer

#Testing purpose using small subset

###convert each chunk into BGE-M3 embeddings.


- Create a small embedding function and test it
- Apply it on the dataset

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_texts(texts, tokenizer, model, device, max_length=512):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    return embeddings.cpu().numpy()

In [ ]:
sample_chunks = train_processed[0]["chunks"][:2]
sample_embeddings = encode_texts(sample_chunks, tokenizer, model, device)

print(sample_embeddings.shape)
print(sample_embeddings[0][:10])

###Prepare chunk records for FAISS. Since each example has multiple chunks

- Need to flatten them into a list like:

   - chunk text
   - question ID / example ID
   - chunk index









In [ ]:
train_chunk_records = []

for example in train_processed:
    ex_id = example["id"]
    question = example["question"]
    answer = example["answer"]

    for chunk_idx, chunk_text in enumerate(example["chunks"]):
        train_chunk_records.append({
            "id": ex_id,
            "question": question,
            "answer": answer,
            "chunk_idx": chunk_idx,
            "chunk_text": chunk_text
        })

print("Total training chunks:", len(train_chunk_records))
print(train_chunk_records[0])

###Store embeddings in FAISS

In [ ]:
# Test on a small subset first
subset_chunks = [record["chunk_text"] for record in train_chunk_records[:1000]]

all_embeddings = []

batch_size = 16

for start_idx in tqdm(range(0, len(subset_chunks), batch_size)):
    batch_chunks = subset_chunks[start_idx:start_idx + batch_size]

    batch_embeddings = encode_texts(
        batch_chunks,
        tokenizer,
        model,
        device
    )

    all_embeddings.append(batch_embeddings)

# Combine into one array
all_embeddings = np.vstack(all_embeddings)

print(all_embeddings.shape)

In [ ]:
# Create FAISS index
embedding_dim = all_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)

# Add embeddings to the index
index.add(all_embeddings)

print("Number of indexed chunks:", index.ntotal)

###Question embedding + retrieval test.

In [ ]:
# Example question
query = train_processed[0]["question"]

# Encode the question
query_embedding = encode_texts([query], tokenizer, model, device)

print("Query embedding shape:", query_embedding.shape)

###Retrieve top-k chunks from FAISS (Facebook AI Similarity Search):

In [ ]:
k = 3

distances, indices = index.search(query_embedding, k)

print("Retrieved indices:", indices)
print("Distances:", distances)

In [ ]:
for rank, idx in enumerate(indices[0], 1):
    print(f"\nRank {rank} | Chunk index: {idx}\n")
    print(train_chunk_records[idx]["chunk_text"][:1000])

###combine the retrieved chunks into one context

In [ ]:
retrieved_context = "\n\n".join(
    [train_chunk_records[idx]["chunk_text"] for idx in indices[0]]
)

print(retrieved_context)

#Load Qwen tokenizer and model

In [ ]:


qwen_model_name = "Qwen/Qwen3-4B-Instruct-2507"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

###Encoder / retriever

- preprocess chunks
- embed with BGE-M3
- store in FAISS


#Training-Phase Baseline RAG Retrieval Setup (Full Dataset):

- Full dataset chunking (778K chunks)
- BGE-M3 embeddings
- FAISS index built on full chunk corpus




In [ ]:
import re
import string

def normalize_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text

In [ ]:
full_train_chunk_records = []

for example in train_processed:
    ex_id = example["id"]
    question = example["question"]
    answer = example["answer"]

    for chunk_idx, chunk_text in enumerate(example["chunks"]):
        full_train_chunk_records.append({
            "id": ex_id,
            "question": question,
            "answer": answer,
            "chunk_idx": chunk_idx,
            "chunk_text": chunk_text
        })

print("Total full training chunks:", len(full_train_chunk_records))
print(full_train_chunk_records[0])

In [ ]:
full_chunk_texts = [record["chunk_text"] for record in full_train_chunk_records]

print("Number of full chunk texts:", len(full_chunk_texts))
print(full_chunk_texts[0][:300])

In [ ]:
full_embeddings = []

batch_size = 64

for start_idx in tqdm(range(0, len(full_chunk_texts), batch_size)):
    batch_chunks = full_chunk_texts[start_idx:start_idx + batch_size]

    batch_embeddings = encode_texts(
        batch_chunks,
        tokenizer,
        model,
        device
    )

    full_embeddings.append(batch_embeddings)

full_embeddings = np.vstack(full_embeddings)

print("Full embedding shape:", full_embeddings.shape)

In [ ]:
embedding_dim = full_embeddings.shape[1]

index_full = faiss.IndexFlatL2(embedding_dim)
index_full.add(full_embeddings)

In [ ]:
faiss.write_index(index_full, "index_full.faiss")
np.save("full_chunk_embeddings.npy", full_embeddings)

import pickle
with open("full_train_chunk_records.pkl", "wb") as f:
    pickle.dump(full_train_chunk_records, f)

In [ ]:
from collections import Counter

def compute_em(prediction, ground_truth):
    return int(normalize_text(prediction) == normalize_text(ground_truth))

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(ground_truth).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
import matplotlib.pyplot as plt

def plot_em_f1_bar(em_score, f1_score, title="Baseline RAG Validation Performance"):
    metrics = ["EM", "F1"]
    values = [em_score, f1_score]

    plt.figure(figsize=(6, 4))
    plt.bar(metrics, values)
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title(title)

    for i, v in enumerate(values):
        plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

    plt.show()

In [ ]:
def run_baseline_rag(question, index, chunk_records, tokenizer, model, device,
                     qwen_tokenizer, qwen_model, k=3, max_new_tokens=50, temperature=0.1):

    query_embedding = encode_texts([question], tokenizer, model, device)
    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = [chunk_records[idx]["chunk_text"] for idx in indices[0]]
    retrieved_context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Answer the question using only the retrieved context.

Return only the exact short answer phrase.
Do not explain.
Do not add dates, years, or extra words.

Context:
{retrieved_context}

Question:
{question}

Short answer:
"""

    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature
        )

    generated_answer = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "retrieved_indices": indices[0],
        "retrieved_chunks": retrieved_chunks,
        "generated_answer": generated_answer
    }

In [ ]:
result = run_baseline_rag(
    question=val_data[0]["question"],
    index=index_full,
    chunk_records=full_train_chunk_records,
    tokenizer=tokenizer,
    model=model,
    device=device,
    qwen_tokenizer=qwen_tokenizer,
    qwen_model=qwen_model,
    k=3
)

In [ ]:
import re

pred = result["generated_answer"]

# clean prediction
pred = pred.strip().split("\n")[0].strip()
pred = re.sub(r"\(.*?\)", "", pred).strip()

print("Prediction:", pred)
print("Gold:", val_data[0]["answer"])
print("EM:", compute_em(pred, val_data[0]["answer"]))
print("F1:", compute_f1(pred, val_data[0]["answer"]))

In [ ]:
import re

em_scores = []
f1_scores = []

for i in range(10):
    question = val_data[i]["question"]
    gold = val_data[i]["answer"]

    result = run_baseline_rag(
        question=question,
        index=index_full,
        chunk_records=full_train_chunk_records,
        tokenizer=tokenizer,
        model=model,
        device=device,
        qwen_tokenizer=qwen_tokenizer,
        qwen_model=qwen_model,
        k=3,
        temperature=0.1,
        max_new_tokens=50
    )

    pred = result["generated_answer"]
    pred = pred.strip().split("\n")[0].strip()
    pred = re.sub(r"\(.*?\)", "", pred).strip()

    em = compute_em(pred, gold)
    f1 = compute_f1(pred, gold)

    em_scores.append(em)
    f1_scores.append(f1)

    print(f"\nExample {i+1}")
    print("Prediction:", pred)
    print("Gold:", gold)
    print("EM:", em)
    print("F1:", f1)


avg_EM = sum(em_scores) / len(em_scores)
avg_F1 = sum(f1_scores) / len(f1_scores)

print("\nAverage EM:", avg_EM)
print("Average F1:", avg_F1)

In [ ]:
import matplotlib.pyplot as plt

def plot_em_f1_bar(em_score, f1_score, title="Baseline RAG Validation Performance"):
    metrics = ["EM", "F1"]
    values = [em_score, f1_score]

    plt.figure(figsize=(6, 4))
    plt.bar(metrics, values)
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title(title)

    for i, v in enumerate(values):
        plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

    plt.show()

The baseline results show that incorrect predictions are primarily due to missing or incorrect retrieved context rather than generation errors. This indicates that retrieval quality is a key limiting factor in the Baseline RAG model.

#Hyperparameter tuning setup:
Before fine-tuning, the Baseline RAG model with the pretrained Qwen generator is evaluated on the validation set, and key hyperparameters are tuned to establish a reliable baseline for later comparison.

- tune k

- tune temperature

- tune max_new_tokens

In [ ]:
import matplotlib.pyplot as plt

def plot_hyperparameter_bar(param_values, em_scores, f1_scores, param_name="k", title=None):
    x = range(len(param_values))
    width = 0.35

    if title is None:
        title = f"Hyperparameter Tuning: {param_name}"

    plt.figure(figsize=(8, 4))
    plt.bar([i - width/2 for i in x], em_scores, width=width, label="EM")
    plt.bar([i + width/2 for i in x], f1_scores, width=width, label="F1")

    plt.xticks(list(x), [str(v) for v in param_values])
    plt.ylim(0, 1)
    plt.xlabel(param_name)
    plt.ylabel("Score")
    plt.title(title)
    plt.legend()

    for i, v in enumerate(em_scores):
        plt.text(i - width/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

    for i, v in enumerate(f1_scores):
        plt.text(i + width/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

    plt.show()

#Evaluation Function for EM and F1 methods

In [ ]:
def evaluate_model(qwen_model, k=3, temperature=0.0, max_new_tokens=20, num_samples=10):

    em_scores = []
    f1_scores = []

    for example in val_data[:num_samples]:
        question = example["question"]
        gold = example["answer"]

        result = run_baseline_rag(
            question=question,
            index=index_full,
            chunk_records=full_train_chunk_records,
            tokenizer=tokenizer,
            model=model,
            device=device,
            qwen_tokenizer=qwen_tokenizer,
            qwen_model=qwen_model,
            k=k,
            temperature=temperature,
            max_new_tokens=max_new_tokens
        )

        pred = result["generated_answer"]

        # clean prediction
        pred = pred.strip().split("\n")[0].strip()
        pred = re.sub(r"\(.*?\)", "", pred).strip()

        em_scores.append(compute_em(pred, gold))
        f1_scores.append(compute_f1(pred, gold))

    avg_em = sum(em_scores) / len(em_scores)
    avg_f1 = sum(f1_scores) / len(f1_scores)

    return avg_em, avg_f1

In [ ]:
def run_baseline_rag(question, index, chunk_records, tokenizer, model, device,
                     qwen_tokenizer, qwen_model, k=3, max_new_tokens=50, temperature=0.1):

    # Encode query
    query_embedding = encode_texts([question], tokenizer, model, device)
    distances, indices = index.search(query_embedding, k)

    # Retrieve context
    retrieved_chunks = [chunk_records[idx]["chunk_text"] for idx in indices[0]]
    retrieved_context = "\n\n".join(retrieved_chunks)

    # Prompt
    prompt = f"""
Answer the question using only the retrieved context.

Return only the exact short answer phrase.
Do not explain.
Do not add dates, years, or extra words.

Context:
{retrieved_context}

Question:
{question}

Short answer:
"""

    # Tokenize
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

    # Generate (FIXED PART)
    with torch.no_grad():
        if temperature == 0.0:
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False   # greedy decoding
            )
        else:
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True
            )

    # Decode output
    generated_answer = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "retrieved_indices": indices[0],
        "retrieved_chunks": retrieved_chunks,
        "generated_answer": generated_answer
    }

In [ ]:
k_values = [1, 3, 5]

em_scores_k = []
f1_scores_k = []

for k in k_values:
    avg_em, avg_f1 = evaluate_model(
        qwen_model=qwen_model,   # pretrained model
        k=k,
        temperature=0.0,
        max_new_tokens=20,
        num_samples=50
    )

    em_scores_k.append(avg_em)
    f1_scores_k.append(avg_f1)

    print(f"\nk = {k}")
    print("Average EM:", avg_em)
    print("Average F1:", avg_f1)

In [ ]:
plot_hyperparameter_bar(
    param_values=k_values,
    em_scores=em_scores_k,
    f1_scores=f1_scores_k,
    param_name="Top-k",
    title="Top-k Tuning (Pretrained Baseline RAG)"
)

In [ ]:
temperatures = [0.0, 0.1, 0.3]

em_scores_temp = []
f1_scores_temp = []

for temp in temperatures:
    avg_em, avg_f1 = evaluate_model(
        qwen_model=qwen_model,
        k=3,
        temperature=temp,
        max_new_tokens=20,
        num_samples=50
    )

    em_scores_temp.append(avg_em)
    f1_scores_temp.append(avg_f1)

    print(f"\nTemperature = {temp}")
    print("Average EM:", avg_em)
    print("Average F1:", avg_f1)

In [ ]:
plot_hyperparameter_bar(
    param_values=temperatures,
    em_scores=em_scores_temp,
    f1_scores=f1_scores_temp,
    param_name="Temperature",
    title="Temperature Tuning (Pretrained Baseline RAG)"
)

In [ ]:
max_tokens_list = [20, 50, 80]

em_scores_tokens = []
f1_scores_tokens = []

for max_tok in max_tokens_list:
    avg_em, avg_f1 = evaluate_model(
        qwen_model=qwen_model,
        k=3,
        temperature=0.3,
        max_new_tokens=max_tok,
        num_samples=50
    )

    em_scores_tokens.append(avg_em)
    f1_scores_tokens.append(avg_f1)

    print(f"\nMax tokens = {max_tok}")
    print("Average EM:", avg_em)
    print("Average F1:", avg_f1)

In [ ]:
plot_hyperparameter_bar(
    param_values=max_tokens_list,
    em_scores=em_scores_tokens,
    f1_scores=f1_scores_tokens,
    param_name="Max Tokens",
    title="Max Tokens Tuning (Pretrained Baseline RAG)"
)

#Hyperparameters - optimal values

Hyperparameters such as top-k retrieval, generation temperature, and maximum output length were tuned on the validation set using the pretrained Baseline RAG model. Based on empirical evaluation,
- Hyperparameter Tuning Summary (Pretrained Baseline RAG)
Top-k (Retrieval Depth):
Performance improves from k=1 to k=3 and then slightly drops at k=5.
 **Optimal: k = 3 (best balance of relevant context and noise)**

- Temperature (Generation Diversity):
Slight improvement as temperature increases.
**Optimal: temperature = 0.3 (highest F1 with stable EM)**

- Max Tokens (Output Length):
Longer outputs degrade performance due to verbosity/noise.
**Optimal: max_tokens = 20 (short, precise answers)**

#Fine-tuning the Pretrained Decoder model QWEN:

For fine-tuning in the RAG settings, the generator model (Qwen) is adapted using supervised fine-tuning (SFT) on HotpotQA training examples. This fine-tuned generator is intended to be used across all three RAG architectures (Baseline, Enhanced, and Hierarchical). Each training instance is converted into an instruction-style prompt containing only the supporting-fact documents as context, followed by the question, while the target output is the gold short answer. This design encourages the model to generate concise answer phrases rather than long explanatory sentences. Using only supporting-fact context provides cleaner supervision and helps the generator learn grounded question answering based on relevant evidence.

Fine-tuning method
Supervised Fine-Tuning (SFT)
Parameter-efficient tuning: LoRA
Goal: make Qwen return short, correct, context-grounded answers for HotpotQA

In [ ]:
def build_qwen_sft_example(example):
    titles = example["context"]["title"]
    sentences = example["context"]["sentences"]
    supporting_titles = set(example["supporting_facts"]["title"])

    supporting_chunks = []

    for title, sent_list in zip(titles, sentences):
        if title in supporting_titles:
            chunk_text = title + ": " + " ".join(sent_list)
            supporting_chunks.append(chunk_text)

    context_text = "\n\n".join(supporting_chunks)

    prompt = f"""Answer the question using only the provided context.

Return only the exact short answer phrase.
Do not explain.
Do not add dates, years, or extra words.

Context:
{context_text}

Question:
{example["question"]}

Short answer:
"""

    return {
        "prompt": prompt,
        "answer": example["answer"]
    }

In [ ]:
sft_train = train_data.map(build_qwen_sft_example)
sft_val = val_data.map(build_qwen_sft_example)

print(sft_train[0]["prompt"][:1200])
print("Answer:", sft_train[0]["answer"])

In [ ]:
def format_for_sft(example):
    text = example["prompt"] + example["answer"].strip()
    return {"text": text}

sft_train_text = sft_train.map(format_for_sft)
sft_val_text = sft_val.map(format_for_sft)

print(sft_train_text[0]["text"][:1200])

In [ ]:
!pip install -q peft trl bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
qwen_model_name = "Qwen/Qwen3-4B-Instruct-2507"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)

if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_ft_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Tokenizer and model loaded.")

In [ ]:
qwen_ft_model = prepare_model_for_kbit_training(qwen_ft_model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
qwen_ft_model = get_peft_model(qwen_ft_model, lora_config)
qwen_ft_model.print_trainable_parameters()

In [ ]:
sft_train_text = sft_train_text.remove_columns(
    [c for c in sft_train_text.column_names if c != "text"]
)
sft_val_text = sft_val_text.remove_columns(
    [c for c in sft_val_text.column_names if c != "text"]
)

print(sft_train_text.column_names)
print(sft_val_text.column_names)
print(sft_train_text[0]["text"][:800])

In [ ]:
from trl import SFTTrainer, SFTConfig

In [ ]:
sft_config = SFTConfig(
    output_dir="qwen_hotpotqa_lora",
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataset_text_field="text",
    packing=False,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=qwen_ft_model,
    args=sft_config,
    train_dataset=sft_train_text,
    eval_dataset=sft_val_text,
    processing_class=qwen_tokenizer,
)

In [ ]:
trainer.train()

##Evaluate the fine-tuned Qwen Baseline RAG on the validation set using the same optimal values from Hyperparameter tuning:
- k = 3
- temperature = 0.3
- max_new_tokens = 20

In [ ]:
ft_val_em, ft_val_f1 = evaluate_model(
    qwen_model=qwen_ft_model,
    k=3,
    temperature=0.3,
    max_new_tokens=20,
    num_samples=50
)

print("Fine-tuned Baseline RAG - Validation Set")
print("Average EM:", ft_val_em)
print("Average F1:", ft_val_f1)

In [ ]:
baseline_val_em, baseline_val_f1 = evaluate_model(
    qwen_model=qwen_model,
    k=3,
    temperature=0.3,
    max_new_tokens=20,
    num_samples=50
)

print("Baseline (Pretrained)")
print(baseline_val_em, baseline_val_f1)

In [ ]:
import pandas as pd

validation_results = pd.DataFrame({
    "Model": ["Pretrained Baseline RAG", "Fine-tuned Baseline RAG"],
    "Dataset": ["Validation", "Validation"],
    "EM": [baseline_val_em, ft_val_em],
    "F1": [baseline_val_f1, ft_val_f1]
})

validation_results

##Validation Comparison Plot - Pretrained Model vs Fine-tuned Model (SFT with LoRA)

In [ ]:
import matplotlib.pyplot as plt

models = ["Pretrained", "Fine-tuned"]

em_values = [baseline_val_em, ft_val_em]
f1_values = [baseline_val_f1, ft_val_f1]

x = range(len(models))
width = 0.35

plt.figure(figsize=(6, 4))

# EM bars
plt.bar([i - width/2 for i in x], em_values, width=width, label="EM")

# F1 bars
plt.bar([i + width/2 for i in x], f1_values, width=width, label="F1")

plt.xticks(x, models)
plt.ylim(0, 1)
plt.xlabel("Model")
plt.ylabel("Score")
plt.title("Validation Performance: Pretrained vs Fine-tuned Baseline RAG")
plt.legend()

# Add values on bars
for i, v in enumerate(em_values):
    plt.text(i - width/2, v + 0.01, f"{v:.3f}", ha="center")

for i, v in enumerate(f1_values):
    plt.text(i + width/2, v + 0.01, f"{v:.3f}", ha="center")

plt.show()

#Evaluate on Test Set

In [ ]:
test_em, test_f1 = evaluate_model(
    qwen_model=qwen_ft_model,
    k=3,
    temperature=0.3,
    max_new_tokens=20,
    num_samples=150

print("Fine-tuned Baseline RAG - Test Set")
print("Average EM:", test_em)
print("Average F1:", test_f1)

In [ ]:
test_results = pd.DataFrame({
    "Model": ["Fine-tuned Baseline RAG"],
    "Dataset": ["Test"],
    "EM": [test_em],
    "F1": [test_f1]
})

test_results

In [ ]:
import matplotlib.pyplot as plt

metrics = ["EM", "F1"]
values = [test_em, test_f1]

plt.figure(figsize=(5, 4))
plt.bar(metrics, values)

plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Test Performance (Fine-tuned Baseline RAG)")

for i, v in enumerate(values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")

plt.show()

In [ ]:
plt.savefig("validation_comparison_plot.png", dpi=300, bbox_inches='tight')

In [ ]:
plt.show()

In [ ]:
from google.colab import files
files.download("validation_comparison_plot.png")

##Save everything and create zip (model, results, notebook)

In [ ]:
import os
import zipfile
import shutil
from google.colab import files

# 1. Save fine-tuned Qwen LoRA model
trainer.model.save_pretrained("qwen_hotpotqa_lora_final")
qwen_tokenizer.save_pretrained("qwen_hotpotqa_lora_final")

print("Saved model files:")
print(os.listdir("qwen_hotpotqa_lora_final"))

# 2. Save result tables
validation_results.to_csv("validation_results.csv", index=False)
test_results.to_csv("test_results.csv", index=False)

# 3. Zip fine-tuned model folder
shutil.make_archive(
    "qwen_hotpotqa_lora_final",
    "zip",
    "qwen_hotpotqa_lora_final"
)

# 4. Zip RAG retrieval artifacts
with zipfile.ZipFile("rag_data.zip", "w") as zipf:
    zipf.write("index_full.faiss")
    zipf.write("full_chunk_embeddings.npy")
    zipf.write("full_train_chunk_records.pkl")

# 5. Zip evaluation result files
with zipfile.ZipFile("baseline_rag_results.zip", "w") as zipf:
    zipf.write("validation_results.csv")
    zipf.write("test_results.csv")

# 6. Check final files
print("\nFinal files created:")
for file in [
    "qwen_hotpotqa_lora_final.zip",
    "rag_data.zip",
    "baseline_rag_results.zip"
]:
    print(file, "exists:", os.path.exists(file))

# 7. Download all important zip files
files.download("qwen_hotpotqa_lora_final.zip")
files.download("rag_data.zip")
files.download("baseline_rag_results.zip")